# 03 - Covariance and PCA on the FPGA (Tier 2)

`scm_unitary` -> `jacobi_evd`, chained by AXI-Stream: the FPGA receives the frame and returns
8 eigenvalues and the 8x8 eigenvector matrix. MUSIC and the peak search still run on the ARM.

1. Bit-exact check of the board output against the integer models (SCM + Jacobi)
2. DOA from the hardware eigenvectors, uncorrelated and coherent sources
3. Host-observed timing: FPGA vs NumPy on the ARM

Upload `tier2_overlay.bit` and `tier2_overlay.hwh` into the same folder as this notebook.

In [ ]:
import os, time, math
import numpy as np
import pynq
from pynq import Overlay, allocate
print("PYNQ", pynq.__version__, "| numpy", np.__version__)

ol = Overlay(os.path.abspath("tier2_overlay.bit"))
print("IP blocks:", list(ol.ip_dict.keys()))
dma = ol.axi_dma_0
scm_ip = ol.scm_unitary_0
jac_ip = ol.jacobi_evd_0

N, M = 8, 128
IN_FRAC, OUT_FRAC = 15, 27
A_FRAC, CS_FRAC = 25, 30
N_OUT = N + N * N

In [ ]:
# ---- Bit-exact integer model of the IP (same as python/scm_bitexact.py) ----
def quantize_input(x):
    q = lambda v: np.clip(np.floor(v * 2**IN_FRAC + 0.5), -2**15, 2**15 - 1).astype(np.int64)
    return q(x.real), q(x.imag)

def scm_model_raw(xr, xi):
    h = N // 2
    a = xr[:h] - xr[::-1][:h]
    b = xi[:h] - xi[::-1][:h]
    yr = np.vstack([xr[:h] + xr[::-1][:h], b])
    yi = np.vstack([xi[:h] + xi[::-1][:h], -a])
    s = yr @ yr.T + yi @ yi.T
    shift = 2 * IN_FRAC + 8 - OUT_FRAC
    return np.clip((s + (1 << (shift - 1))) >> shift, -2**31, 2**31 - 1)

def pack(xr, xi):
    # one complex sample per word, [15:0] re, [31:16] im, snapshot-major
    return (xr.T.astype(np.uint16).astype(np.uint32)
            | (xi.T.astype(np.uint16).astype(np.uint32) << 16)).ravel()

in_buf = allocate(shape=(N * M,), dtype=np.uint32)
out_buf = allocate(shape=(N_OUT,), dtype=np.uint32)

def tier2_fpga_raw(words):
    """Frame -> (eigenvalues raw Q7.25, V raw Q2.30), computed on the FPGA."""
    in_buf[:] = words
    dma.recvchannel.transfer(out_buf)
    jac_ip.write(0x00, 0x01)              # ap_start (waits for its input stream)
    scm_ip.write(0x00, 0x01)
    dma.sendchannel.transfer(in_buf)
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    o = np.array(out_buf).view(np.int32).astype(np.int64)
    return o[:N], o[N:].reshape(N, N)

def normalize(x):
    return x / (1.001 * max(np.abs(x.real).max(), np.abs(x.imag).max(), 1e-12))

In [ ]:
# ---- Bit-exact integer model of jacobi_evd (same as python/jacobi_bitexact.py) ----
ANG_FRAC, N_CORDIC, NORM_BITS = 28, 24, 30
ATAN = [int(round(math.atan(2.0 ** -i) * 2 ** ANG_FRAC)) for i in range(N_CORDIC)]
KINV = 1.0
for i in range(N_CORDIC):
    KINV /= math.sqrt(1.0 + 2.0 ** (-2 * i))
KINV = int(round(KINV * 2 ** CS_FRAC))
PAIRS = [(p, q) for p in range(N - 1) for q in range(p + 1, N)]

def rnd(x, sh):
    return (x + (1 << (sh - 1))) >> sh

def vectoring(x, y):
    m = max(abs(x), abs(y))
    if m == 0:
        return 0
    sh = NORM_BITS - m.bit_length()
    if sh > 0:
        x, y = x << sh, y << sh
    elif sh < 0:
        x, y = x >> -sh, y >> -sh
    z = 0
    for i in range(N_CORDIC):
        if y >= 0:
            x, y, z = x + (y >> i), y - (x >> i), z + ATAN[i]
        else:
            x, y, z = x - (y >> i), y + (x >> i), z - ATAN[i]
    return z

def rotation(th):
    x, y, z = KINV, 0, th
    for i in range(N_CORDIC):
        if z >= 0:
            x, y, z = x - (y >> i), y + (x >> i), z - ATAN[i]
        else:
            x, y, z = x + (y >> i), y - (x >> i), z + ATAN[i]
    return x, y

def jacobi_model_raw(r_raw, n_sweeps=4):
    a = [[rnd(int(v), 2) for v in row] for row in r_raw]
    v = [[(1 << CS_FRAC) if i == j else 0 for j in range(N)] for i in range(N)]
    for _ in range(n_sweeps):
        for p, q in PAIRS:
            app, aqq, apq = a[p][p], a[q][q], a[p][q]
            x, y = aqq - app, 2 * apq
            if x < 0:
                x, y = -x, -y
            c, s = rotation(vectoring(x, y) >> 1)
            c2, s2, cs = rnd(c * c, 30), rnd(s * s, 30), rnd(c * s, 30)
            for k in range(N):
                if k != p and k != q:
                    akp, akq = a[k][p], a[k][q]
                    a[k][p] = a[p][k] = rnd(c * akp - s * akq, 30)
                    a[k][q] = a[q][k] = rnd(s * akp + c * akq, 30)
                vkp, vkq = v[k][p], v[k][q]
                v[k][p] = rnd(c * vkp - s * vkq, 30)
                v[k][q] = rnd(s * vkp + c * vkq, 30)
            a[p][p] = rnd(c2 * app - 2 * cs * apq + s2 * aqq, 30)
            a[q][q] = rnd(s2 * app + 2 * cs * apq + c2 * aqq, 30)
            a[p][q] = a[q][p] = 0
    return np.array([a[i][i] for i in range(N)], dtype=np.int64), np.array(v, dtype=np.int64)

def to_float(d_raw, v_raw):
    d = d_raw * 2.0 ** -A_FRAC
    order = np.argsort(d)[::-1]
    return d[order], (v_raw * 2.0 ** -CS_FRAC)[:, order]

In [ ]:
# ---- Test 1: bit-exact on 20 random frames (the ARM model takes ~0.1-0.3 s per frame) ----
rng = np.random.RandomState(1)
fails = 0
for t in range(20):
    x = normalize(rng.randn(N, M) + 1j * rng.randn(N, M))
    xr, xi = quantize_input(x)
    d_hw, v_hw = tier2_fpga_raw(pack(xr, xi))
    d_ref, v_ref = jacobi_model_raw(scm_model_raw(xr, xi))
    if not (np.array_equal(d_hw, d_ref) and np.array_equal(v_hw, v_ref)):
        fails += 1
        if fails <= 3:
            print("frame", t, "max |diff| eig", np.abs(d_hw - d_ref).max(), "vec", np.abs(v_hw - v_ref).max())
print("bit-exact frames: %d / 20 %s" % (20 - fails, "PASS" if fails == 0 else "FAIL"))

# Accuracy of the hardware EVD against LAPACK on the same covariance
R = scm_model_raw(xr, xi) * 2.0 ** -OUT_FRAC
lam, V = to_float(d_hw, v_hw)
print("max |eig_FPGA - eigh| = %.2e" % np.abs(lam - np.sort(np.linalg.eigvalsh(R))[::-1]).max())
print("max |V^T V - I|       = %.2e" % np.abs(V.T @ V - np.eye(N)).max())

In [ ]:
# ---- Test 2: DOA from the hardware covariance (unitary MUSIC on the ARM) ----
pos = (np.arange(N) - (N - 1) / 2) * 0.5          # centred virtual ULA, d = lambda/2
grid = np.arange(-90.0, 90.5, 0.5)

def steering(theta_deg):
    return np.exp(1j * 2 * np.pi * np.outer(pos, np.sin(np.deg2rad(np.atleast_1d(theta_deg)))))

h = N // 2
I, J = np.eye(h), np.fliplr(np.eye(h))
Q = np.block([[I, 1j * I], [J, -1j * J]]) / np.sqrt(2)
B = (Q.conj().T @ steering(grid)).real               # real steering table

def simulate(theta, snr_db, coherent=False):
    k = len(theta)
    s = (rng.randn(k, M) + 1j * rng.randn(k, M)) / np.sqrt(2)
    if coherent:
        s = np.repeat(s[:1], k, axis=0) * np.exp(1j * rng.uniform(0, 2 * np.pi, (k, 1)))
    n = 10 ** (-snr_db / 20) * (rng.randn(N, M) + 1j * rng.randn(N, M)) / np.sqrt(2)
    return steering(theta) @ s + n

def music_peaks(Es, k):
    d = np.sum(B**2, axis=0) - np.sum((Es.T @ B)**2, axis=0)
    p = 10 * np.log10(1.0 / np.maximum(d, 1e-12))
    mins = [i for i in range(1, len(d) - 1) if d[i] < d[i-1] and d[i] <= d[i+1]]
    mins = sorted(mins, key=lambda i: d[i])[:k]
    return np.sort(grid[mins]), p

true = np.array([-10.0, 5.0])
results = {}
for coherent in (False, True):
    x = normalize(simulate(true, 10, coherent))
    xr, xi = quantize_input(x)
    lam, V = to_float(*tier2_fpga_raw(pack(xr, xi)))
    est, spec = music_peaks(V[:, :2], 2)          # signal subspace from the FPGA
    results[coherent] = spec
    print("%-12s true %s  ->  estimated %s deg" % ("coherent:" if coherent else "uncorrelated:", true, est))

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 3.5))
for coherent, spec in results.items():
    plt.plot(grid, spec - spec.max(), label="coherent" if coherent else "uncorrelated")
for t in true:
    plt.axvline(t, color="gray", ls=":")
plt.xlabel("angle (deg)"); plt.ylabel("MUSIC spectrum (dB)")
plt.title("Unitary MUSIC from the FPGA eigenvectors, N=8, M=128, SNR 10 dB")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ---- Test 3: host-observed time, FPGA vs NumPy on the ARM (median of 200) ----
x = normalize(rng.randn(N, M) + 1j * rng.randn(N, M))
xr, xi = quantize_input(x)
words = pack(xr, xi)
xq = (xr + 1j * xi) * 2.0**-IN_FRAC

def median_us(fn, reps=200):
    t = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); t.append(time.perf_counter() - t0)
    return 1e6 * np.median(t)

def numpy_scm_evd():
    y = Q.conj().T @ xq
    return np.linalg.eigh((y @ y.conj().T).real / M)

t_fpga = median_us(lambda: tier2_fpga_raw(words))
t_np = median_us(numpy_scm_evd)
t_model = median_us(lambda: jacobi_model_raw(scm_model_raw(xr, xi)), reps=5)
print("FPGA SCM + Jacobi (DMA + IPs, host-observed): %9.1f us" % t_fpga)
print("NumPy float SCM + eigh (LAPACK), ARM        : %9.1f us" % t_np)
print("Integer model SCM + Jacobi, ARM (Python)    : %9.1f us" % t_model)
print("HLS estimate of the IPs alone: see hls/scm/reports and hls/jacobi/reports")